# Gemma4NPC DPO (Direct Preference Optimization)

This notebook aligns the `Gemma4NPC-it` model to prefer natural, engaging, and in-character responses using DPO.

In [ ]:
from unsloth import FastModel, PatchDPOTrainer
PatchDPOTrainer()
import torch

# Load the SFT merged model
model, tokenizer = FastModel.from_pretrained(
    model_name = "outputs/Gemma4NPC-it/merged_float16",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastModel.get_peft_model(
    model,
    r = 8,  # Lower rank for DPO
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 8,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="data/dpo/preference_pairs.jsonl", split="train")

# Format for DPO (requires 'prompt', 'chosen', 'rejected' columns)
def format_dpo(example):
    return {
        "prompt": example["prompt"],
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}]
    }

dataset = dataset.map(format_dpo)

In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth handles implicit reference model
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = DPOConfig(
        output_dir = "outputs/Gemma4NPC-it-DPO",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 5e-7, # Very low LR for DPO
        num_train_epochs = 1,
        beta = 0.1, # KL penalty
        bf16 = True,
        logging_steps = 5,
    ),
)

trainer.train()

In [ ]:
model.save_pretrained_merged("outputs/Gemma4NPC-it-DPO/merged_float16", tokenizer, save_method="merged_16bit")